# Phần 3: Tổng hợp và thống kê (Aggregation)
Sau khi hoàn thành bước làm sạch và biến đổi dữ liệu, dữ liệu được đưa vào bước Aggregation bằng Apache Spark. Mục tiêu là tổng hợp dữ liệu theo nhiều nhóm khác nhau để tạo ra các chỉ số phục vụ phân tích đơn hàng, doanh thu và hiệu suất giao hàng.

**Lưu ý khi chạy:** chạy lần lượt từng cell từ trên xuống, đợi cell xong rồi mới chạy cell tiếp. Kernel phải là Python của `venv`.

## 0. Cấu hình môi trường (chạy cell này TRƯỚC khi tạo Spark)

In [35]:
import os, sys, time
from pathlib import Path
# common.py cấu hình Java (đường dẫn riêng từng máy nằm trong local_config.py) và phải được nạp TRƯỚC khi tạo Spark
from common import get_spark, TRANSFORMED_DIR, BASE, RAW_CSV

print("Python đang dùng:", sys.executable)
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))

Python đang dùng: d:\learn-CODE\baocao_thuyettrinh\Bigdata\file_claude\file_etracted\giua-ki\giua-ki\venv\Scripts\python.exe
JAVA_HOME: D:\DOWNLOAD\OpenJDK17U-jdk_x64_windows_hotspot_17.0.20.1_1\jdk-17.0.20.1+1


## 1. Khởi tạo Spark và đọc dữ liệu

In [3]:
from pyspark.sql import functions as F

spark = get_spark("BigDataProject")
print("Spark version:", spark.version)

Spark version: 4.3.0


Khởi tạo Spark. Thư viện `pyspark.sql.functions` được dùng để thực hiện các phép tính như đếm, tính tổng, trung bình và sắp xếp dữ liệu.

In [4]:
# Đọc dữ liệu chi tiết từ kho Parquet (kết quả bước Transformation), cache lại vì sẽ tổng hợp nhiều lần
df = spark.read.parquet(str(TRANSFORMED_DIR)).cache()
print("So dong:", df.count())
print("So cot:", len(df.columns))
df.printSchema()
df.show(10, truncate=False)

So dong: 100002
So cot: 33
root
 |-- Order_ID: integer (nullable = true)
 |-- User_ID: string (nullable = true)
 |-- Restaurant_ID: integer (nullable = true)
 |-- Driver_ID: integer (nullable = true)
 |-- Item_Name: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Total_Price: double (nullable = true)
 |-- Order_Time: string (nullable = true)
 |-- Delivery_Time: string (nullable = true)
 |-- Delivery_Duration_Minutes: integer (nullable = true)
 |-- City: string (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- Order_Status: string (nullable = true)
 |-- Driver_Vehicle: string (nullable = true)
 |-- Restaurant_Lat: double (nullable = true)
 |-- Restaurant_Lon: double (nullable = true)
 |-- Customer_Lat: double (nullable = true)
 |-- Customer_Lon: double (nullable = true)
 |-- Driver_Lat: double (nullable = true)
 |-- Driver_Lon: double (nullable = true)
 |-- Delivery_Distance_km: double (nullable = true)
 |-- Traffic_Level: string (nullable = tru

Dữ liệu được đọc từ kho Parquet `data_parquet/transformed/` (kết quả bước Transformation, phân vùng theo `Order_Month`). Parquet giữ nguyên kiểu dữ liệu nên không cần suy luận lại như CSV. Các lệnh kiểm tra số dòng, số cột, kiểu dữ liệu và vài bản ghi đầu giúp xác nhận dữ liệu đầu vào hợp lệ trước khi tổng hợp.

## KPI tổng quan

In [5]:
from aggregations import build_tables  # nơi duy nhất định nghĩa phép tổng hợp, dùng chung với save_parquet.py
tables = build_tables(df)
tables["overview"].show(truncate=False)

+------------+-------------+---------------+----------------------+-------------------+--------------+
|Total_Orders|Total_Revenue|Average_Revenue|Average_Delivery_Hours|Average_Distance_km|Total_Quantity|
+------------+-------------+---------------+----------------------+-------------------+--------------+
|100002      |2.689280561E7|268.92         |0.625                 |2.17               |299116        |
+------------+-------------+---------------+----------------------+-------------------+--------------+



Tạo các chỉ số tổng quan cho toàn bộ tập dữ liệu gồm tổng số đơn hàng, tổng doanh thu, doanh thu trung bình, thời gian giao hàng trung bình, khoảng cách giao hàng trung bình và tổng số lượng sản phẩm. Các chỉ số này giúp có cái nhìn tổng quát về quy mô và tình hình hoạt động của dữ liệu.

## Bảng 1: Thống kê theo thành phố

In [6]:
tables["city_summary"].show(20, truncate=False)

+----------+------------+-------------+---------------+----------------------+---------------------+
|City      |Total_Orders|Total_Revenue|Average_Revenue|Average_Delivery_Hours|Revenue_Share_Percent|
+----------+------------+-------------+---------------+----------------------+---------------------+
|Zagazig   |14538       |3905102.65   |268.61         |0.628                 |14.52                |
|Tanta     |14336       |3896972.63   |271.83         |0.625                 |14.49                |
|Cairo     |14382       |3856412.5    |268.14         |0.624                 |14.34                |
|Alexandria|14168       |3841624.98   |271.15         |0.624                 |14.28                |
|Assiut    |14302       |3839095.0    |268.43         |0.627                 |14.28                |
|Giza      |14232       |3798249.8    |266.88         |0.623                 |14.12                |
|Mansoura  |14044       |3755348.05   |267.4          |0.626                 |13.96        

Dùng `groupBy("City")` để nhóm đơn hàng theo thành phố, rồi tính tổng số đơn, tổng doanh thu, doanh thu trung bình và thời gian giao trung bình. Kết quả giúp xác định sự khác biệt về hoạt động kinh doanh giữa các thành phố.

### Xếp hạng và tỷ trọng doanh thu theo thành phố (Window function)

In [23]:
from pyspark.sql.window import Window

# Hàm cửa sổ (Window) xếp hạng ngay trong Spark
w_rev = Window.orderBy(F.col("Total_Revenue").desc())
w_ord = Window.orderBy(F.col("Total_Orders").desc())
# Hàm cửa sổ (Window) xếp hạng ngay trong Spark, thay cho việc sắp xếp + limit(10) trên các bảng top riêng lẻ
w_rev = Window.orderBy(F.col("Total_Revenue").desc())
w_ord = Window.orderBy(F.col("Total_Orders").desc())

city_rank = (
    tables["city_revenue_analysis"]          # đã có sẵn cột Revenue_Share_Percent
    .withColumn("Rank_Revenue", F.rank().over(w_rev))
    .withColumn("Rank_Orders", F.rank().over(w_ord))
    .orderBy("Rank_Revenue")
)
city_rank.show(20, truncate=False)
tables["city_ranking"] = city_rank           # để được lưu cùng các bảng khác ở cuối notebook

# Độ chênh lệch giữa các thành phố (bảng nhỏ 7 dòng nên chuyển sang pandas để tính là an toàn)
p = city_rank.toPandas()
for c in ["Total_Orders", "Total_Revenue"]:
    mx, mn, mean = p[c].max(), p[c].min(), p[c].mean()
    print(f"{c}: cao nhất {mx:,.0f} | thấp nhất {mn:,.0f} | chênh {(mx - mn) / mean:.1%} so với trung bình")

+----------+------------+-------------+---------------+----------------------+---------------------+------------+-----------+
|City      |Total_Orders|Total_Revenue|Average_Revenue|Average_Delivery_Hours|Revenue_Share_Percent|Rank_Revenue|Rank_Orders|
+----------+------------+-------------+---------------+----------------------+---------------------+------------+-----------+
|Zagazig   |14538       |3905102.65   |268.61         |0.628                 |14.52                |1           |1          |
|Tanta     |14336       |3896972.63   |271.83         |0.625                 |14.49                |2           |3          |
|Cairo     |14382       |3856412.5    |268.14         |0.624                 |14.34                |3           |2          |
|Alexandria|14168       |3841624.98   |271.15         |0.624                 |14.28                |4           |6          |
|Assiut    |14302       |3839095.0    |268.43         |0.627                 |14.28                |5           |4    

Nhận xét: doanh thu và số đơn giữa các thành phố chênh nhau rất ít (vài phần trăm so với trung bình, tỷ trọng quanh 14%), và thứ hạng theo doanh thu khác thứ hạng theo số đơn do doanh thu trung bình mỗi đơn khác nhau nhẹ. Mức đều đặn này phù hợp với việc dataset là dữ liệu tổng hợp (`synthetic`), nên không kết luận thành phố nào vượt trội.

## Bảng 2: Thống kê theo phân khúc đơn hàng (Order Segment)

In [10]:
tables["segment_summary"].show(truncate=False)

+-------------+------------+-------------+---------------+----------------+
|order_segment|Total_Orders|Total_Revenue|Average_Revenue|Average_Quantity|
+-------------+------------+-------------+---------------+----------------+
|Medium       |44838       |1.448366146E7|323.02         |3.53            |
|High         |12320       |7375727.23   |598.68         |4.66            |
|Low          |42844       |5033416.92   |117.48         |1.95            |
+-------------+------------+-------------+---------------+----------------+



## Bảng 3: Thống kê hiệu suất giao hàng (Delivery Category)

In [11]:
tables["delivery_summary"].show(truncate=False)

+-----------------+------------+----------------------+-------------------+-------------+
|Delivery_Category|Total_Orders|Average_Delivery_Hours|Average_Distance_km|Total_Revenue|
+-----------------+------------+----------------------+-------------------+-------------+
|Normal           |48411       |0.634                 |2.16               |1.297401602E7|
|Fast             |27414       |0.417                 |2.17               |7399720.02   |
|Slow             |24177       |0.845                 |2.16               |6519069.57   |
+-----------------+------------+----------------------+-------------------+-------------+



## Bảng 4: Thống kê theo khoảng cách (Distance Category)

In [12]:
tables["distance_summary"].show(truncate=False)

+-----------------+------------+-------------------+----------------------+-------------+
|Distance_Category|Total_Orders|Average_Distance_km|Average_Delivery_Hours|Total_Revenue|
+-----------------+------------+-------------------+----------------------+-------------+
|Near             |77069       |1.74               |0.625                 |2.069101174E7|
|Medium           |22779       |3.58               |0.626                 |6160063.27   |
|Far              |154         |5.16               |0.618                 |41730.6      |
+-----------------+------------+-------------------+----------------------+-------------+



## Bảng 5: Thống kê theo giờ đặt hàng (Order Hour)

In [13]:
tables["hour_summary"].show(24, truncate=False)

+----------+------------+-------------+---------------+----------------------+
|Order_Hour|Total_Orders|Total_Revenue|Average_Revenue|Average_Delivery_Hours|
+----------+------------+-------------+---------------+----------------------+
|0         |4157        |1101177.74   |264.9          |0.622                 |
|1         |4256        |1157358.28   |271.94         |0.627                 |
|2         |4179        |1113181.26   |266.38         |0.628                 |
|3         |4163        |1146980.65   |275.52         |0.628                 |
|4         |4263        |1137969.29   |266.94         |0.622                 |
|5         |4111        |1092768.19   |265.82         |0.63                  |
|6         |4168        |1139597.63   |273.42         |0.627                 |
|7         |4174        |1120522.27   |268.45         |0.627                 |
|8         |4164        |1117493.34   |268.37         |0.622                 |
|9         |4153        |1105769.42   |266.26       

### Top 10 giờ có nhiều đơn đặt hàng nhất

In [14]:
tables["top_hours"].show(truncate=False)

+----------+------------+-------------+---------------+----------------------+
|Order_Hour|Total_Orders|Total_Revenue|Average_Revenue|Average_Delivery_Hours|
+----------+------------+-------------+---------------+----------------------+
|4         |4263        |1137969.29   |266.94         |0.622                 |
|1         |4256        |1157358.28   |271.94         |0.627                 |
|14        |4246        |1149039.16   |270.62         |0.623                 |
|16        |4239        |1142388.01   |269.49         |0.622                 |
|18        |4225        |1130830.84   |267.65         |0.626                 |
|15        |4216        |1133602.52   |268.88         |0.623                 |
|23        |4210        |1130713.8    |268.58         |0.628                 |
|11        |4192        |1117064.84   |266.48         |0.623                 |
|12        |4186        |1124177.43   |268.56         |0.626                 |
|2         |4179        |1113181.26   |266.38       

## Bảng 6: Truy vấn bằng Spark SQL
Ngoài DataFrame API, Spark cho phép viết SQL trực tiếp trên dữ liệu sau khi đăng ký thành bảng tạm (`createOrReplaceTempView`). Cùng một engine, cùng kết quả.

In [24]:
df.createOrReplaceTempView("orders")

spark.sql("""
    SELECT City,
           Delivery_Category,
           COUNT(*)                                 AS orders,
           ROUND(AVG(Delivery_Duration_Hours), 3)   AS avg_hours,
           ROUND(SUM(revenue), 2)                   AS total_revenue
    FROM orders
    GROUP BY City, Delivery_Category
    ORDER BY City, orders DESC
""").show(30, truncate=False)

+----------+-----------------+------+---------+-------------+
|City      |Delivery_Category|orders|avg_hours|total_revenue|
+----------+-----------------+------+---------+-------------+
|Alexandria|Normal           |6847  |0.634    |1855008.43   |
|Alexandria|Fast             |3929  |0.416    |1066973.42   |
|Alexandria|Slow             |3392  |0.845    |919643.13    |
|Assiut    |Normal           |6940  |0.634    |1851632.56   |
|Assiut    |Fast             |3855  |0.417    |1033340.59   |
|Assiut    |Slow             |3507  |0.845    |954121.85    |
|Cairo     |Normal           |7011  |0.633    |1878867.57   |
|Cairo     |Fast             |3929  |0.416    |1060007.72   |
|Cairo     |Slow             |3442  |0.844    |917537.21    |
|Giza      |Normal           |6885  |0.632    |1813175.78   |
|Giza      |Fast             |3951  |0.417    |1066123.78   |
|Giza      |Slow             |3396  |0.845    |918950.24    |
|Mansoura  |Normal           |6782  |0.635    |1819922.19   |
|Mansour

## Bảng 7: Pivot — số đơn theo thành phố và hiệu suất giao hàng
`pivot` xoay giá trị của một cột thành các cột, rất tiện để vẽ biểu đồ cột nhóm.

In [25]:
city_delivery = (
    df.groupBy("City")
      .pivot("Delivery_Category", ["Fast", "Normal", "Slow"])   # liệt kê sẵn giá trị để Spark không phải quét thêm một lượt
      .count()
      .orderBy("City")
)
city_delivery.show(truncate=False)
tables["city_delivery_pivot"] = city_delivery

+----------+----+------+----+
|City      |Fast|Normal|Slow|
+----------+----+------+----+
|Alexandria|3929|6847  |3392|
|Assiut    |3855|6940  |3507|
|Cairo     |3929|7011  |3442|
|Giza      |3951|6885  |3396|
|Mansoura  |3859|6782  |3403|
|Tanta     |3999|6875  |3462|
|Zagazig   |3892|7071  |3575|
+----------+----+------+----+



## Bảng 8: Chứng minh lợi ích của phân vùng (partition pruning)
Dữ liệu `transformed/` được chia thành các thư mục `Order_Day=<ngày>/`. Khi lọc theo `Order_Day`, Spark chỉ cần đọc đúng thư mục của ngày đó và bỏ qua các ngày còn lại. Kết quả `explain()` có dòng `PartitionFilters` là bằng chứng.

In [26]:
pq = spark.read.parquet(str(TRANSFORMED_DIR))      # đọc mới, không dùng bản đã cache để plan hiện rõ phần đọc file
q = pq.filter("Order_Day = 5")
q.explain()
print("Số đơn ngày 5:", q.count(), "| tổng số đơn:", pq.count())

== Physical Plan ==
*(1) Filter (isnotnull(Order_Day#23708) AND (Order_Day#23708 = 5))
+- InMemoryTableScan [Order_ID#23676, User_ID#23677, Restaurant_ID#23678, Driver_ID#23679, Item_Name#23680, Quantity#23681, Total_Price#23682, Order_Time#23683, Delivery_Time#23684, Delivery_Duration_Minutes#23685, City#23686, Payment_Method#23687, Order_Status#23688, Driver_Vehicle#23689, Restaurant_Lat#23690, Restaurant_Lon#23691, Customer_Lat#23692, Customer_Lon#23693, Driver_Lat#23694, Driver_Lon#23695, Delivery_Distance_km#23696, Traffic_Level#23697, Driver_Availability#23698, revenue#23699, order_segment#23700, ... 8 more fields], [isnotnull(Order_Day#23708), (Order_Day#23708 = 5)]
      +- InMemoryRelation [Order_ID#23676, User_ID#23677, Restaurant_ID#23678, Driver_ID#23679, Item_Name#23680, Quantity#23681, Total_Price#23682, Order_Time#23683, Delivery_Time#23684, Delivery_Duration_Minutes#23685, City#23686, Payment_Method#23687, Order_Status#23688, Driver_Vehicle#23689, Restaurant_Lat#23690, 

Trong kết quả `explain()` hãy tìm `PartitionFilters: [..., (Order_Day = 5)]`. Nếu dòng này không xuất hiện, kiểm tra lại `transformation.py` đã ghi với `partition_by=["Order_Day"]` chưa.

## Bảng 9: So sánh CSV và Parquet
So sánh kích thước và thời gian chạy cùng một phép tổng hợp trên hai định dạng. Với 100.000 dòng sự khác biệt sẽ nhỏ; lợi ích của Parquet (nén, lưu theo cột, giữ kiểu dữ liệu) rõ hơn khi dữ liệu lớn.

In [36]:
# Hàm tính kích thước thư mục (tính bằng MB)
def dir_size_mb(directory):
    directory = Path(directory)
    total_size = sum(f.stat().st_size for f in directory.glob('**/*') if f.is_file())
    return total_size / 1024 / 1024

def avg_time(fn, runs=3):
    fn()                                    # chạy khởi động, bỏ qua (JVM, cache hệ thống)
    ts = []
    for _ in range(runs):
        t = time.perf_counter(); fn(); ts.append(time.perf_counter() - t)
    return sum(ts) / len(ts)

csv_df = spark.read.option("header", "true").option("inferSchema", "true").csv(str(RAW_CSV))
pq_df  = spark.read.parquet(str(TRANSFORMED_DIR))

job = lambda d: d.groupBy("City").agg(F.sum("Total_Price").alias("rev")).collect()
t_csv, t_pq = avg_time(lambda: job(csv_df)), avg_time(lambda: job(pq_df))

print(f"Kích thước CSV gốc      : {os.path.getsize(RAW_CSV) / 1024 / 1024:.2f} MB")
print(f"Kích thước Parquet      : {dir_size_mb(TRANSFORMED_DIR):.2f} MB (33 cột, đã nén)")
print(f"Thời gian tổng hợp CSV  : {t_csv:.2f} giây")
print(f"Thời gian tổng hợp Parquet: {t_pq:.2f} giây")

Kích thước CSV gốc      : 19.28 MB
Kích thước Parquet      : 8.92 MB (33 cột, đã nén)
Thời gian tổng hợp CSV  : 0.28 giây
Thời gian tổng hợp Parquet: 0.13 giây


## Lưu lại các bảng để phân tích

In [37]:
out_dir = os.path.join(BASE, "output")
os.makedirs(out_dir, exist_ok=True)

# Các bảng tổng hợp rất nhỏ (vài chục dòng) nên chuyển sang pandas để xuất Excel là an toàn.
# Dữ liệu chi tiết (hàng trăm nghìn dòng) KHÔNG đi qua pandas.
for name, t in tables.items():
    t.toPandas().to_excel(os.path.join(out_dir, name + ".xlsx"), index=False)

print("Đã lưu toàn bộ kết quả Aggregation vào thư mục:", out_dir)
print("Các file:", sorted(os.listdir(out_dir)))

Đã lưu toàn bộ kết quả Aggregation vào thư mục: D:\learn-CODE\baocao_thuyettrinh\Bigdata\file_claude\file_etracted\giua-ki\giua-ki\output
Các file: ['city_delivery_pivot.xlsx', 'city_ranking.xlsx', 'city_revenue_analysis.xlsx', 'city_summary.xlsx', 'delivery_summary.xlsx', 'distance_summary.xlsx', 'hour_summary.xlsx', 'overview.xlsx', 'segment_summary.xlsx', 'top_city_orders.xlsx', 'top_city_revenue.xlsx', 'top_hours.xlsx']


Các bảng tổng hợp được lưu dưới định dạng Excel để dễ xem. Bản Parquet của 5 bảng cốt lõi do `save_parquet.py` tạo ra từ **cùng hàm** `build_tables()`, nên số liệu hai nơi luôn giống nhau.